# Skill loading, studied

How Claude Code loads a skill, reproduced here with `bro_skills/`:

1. **Discovery** — scan every `bro_skills/*/SKILL.md`, parse only the YAML frontmatter (`name`, `description`). This is cheap and happens for *all* skills up front — like the skill listing Claude sees in its system prompt.
2. **Selection** — pick a skill by matching intent against the descriptions (here: just by name, to keep it simple).
3. **Load** — read the *full* SKILL.md body only for the chosen skill. This is the "progressive disclosure" part: full instructions are loaded lazily, not for every skill.
4. **Use** — feed the loaded instructions to an LLM call (or in this notebook, just print them, since no LLM call is wired up yet).

The plumbing this notebook exercises now lives in `src/bro_agent/` (see the
package's own docstrings for detail) so the sales-funnel study in
`sales_agent.ipynb` can reuse it without copy-pasting cells. Each cell here
imports one small piece and demonstrates it.

In [1]:
from bro_agent.skills import SKILLS_DIR, discover_skills, load_skill, skill_dir

skill_registry = discover_skills()
skill_registry

{'close-handoff': {'description': 'Terminal stage once a lead has agreed to move forward. Creates a handoff for a human rep -- never collects payment or signs anything itself. Use this while a lead is in the close_handoff stage of the funnel.',
  'stage': 'close_handoff',
  'path': WindowsPath('D:/study-on-agent/bro_skills/close-handoff/SKILL.md')},
 'grep-file': {'description': 'Search for a text pattern inside a specific file and return matching lines with their line numbers. Use this when the user names a file and a pattern/word/string to find within it.',
  'stage': None,
  'path': WindowsPath('D:/study-on-agent/bro_skills/grep-file/SKILL.md')},
 'handle-objection': {'description': 'Address a specific concern or pushback (price, fit, timing) a lead has raised. Use this while a lead is in the handle_objection stage of the funnel.',
  'stage': 'handle_objection',
  'path': WindowsPath('D:/study-on-agent/bro_skills/handle-objection/SKILL.md')},
 'hello-world': {'description': 'A minim

## Stage 2 — load a chosen skill's full instructions

`skill_registry` only has name + description. Loading the body happens
only for the one skill actually picked -- that's the lazy part.

In [2]:
instructions = load_skill("hello-world", skill_registry)
print(instructions)

# Hello World

When invoked, do the following:

1. Greet the person by name, warmly and briefly.
2. In one sentence, explain that you were invoked as the `hello-world` skill.

Keep the whole response to two short sentences. Do not add anything else.


In [3]:
instructions = load_skill("handle-objection", skill_registry)
print(instructions)

# Handle Objection

The customer raised a concern. Address it directly and honestly — don't be
pushy, and don't invent facts (pricing or product claims) to overcome the
objection; if you need a fact, that belongs in `nurture` or `present_offer`.

1. If the objection is about price or plan fit, propose moving back to
   `present_offer` once addressed.
2. If the objection is about the product itself (does it do X?), propose
   moving back to `nurture`.
3. If the objection isn't resolved, stay in `handle_objection`.
4. Never propose `close_handoff` from here directly — resolve the
   objection first by returning to `present_offer` or `nurture`.


## Interactive skills — tools the kernel can actually run

A skill is instructions, not code. `grep-file/SKILL.md` tells the agent to
call `grep_file(pattern, path)` — here are those tools, and re-running
discovery now that a second skill exists.

In [4]:
from bro_agent.execution import TOOLS, read_file, grep_file

# re-discover now that bro_skills/ has a second skill
skill_registry = discover_skills()
skill_registry

{'close-handoff': {'description': 'Terminal stage once a lead has agreed to move forward. Creates a handoff for a human rep -- never collects payment or signs anything itself. Use this while a lead is in the close_handoff stage of the funnel.',
  'stage': 'close_handoff',
  'path': WindowsPath('D:/study-on-agent/bro_skills/close-handoff/SKILL.md')},
 'grep-file': {'description': 'Search for a text pattern inside a specific file and return matching lines with their line numbers. Use this when the user names a file and a pattern/word/string to find within it.',
  'stage': None,
  'path': WindowsPath('D:/study-on-agent/bro_skills/grep-file/SKILL.md')},
 'handle-objection': {'description': 'Address a specific concern or pushback (price, fit, timing) a lead has raised. Use this while a lead is in the handle_objection stage of the funnel.',
  'stage': 'handle_objection',
  'path': WindowsPath('D:/study-on-agent/bro_skills/handle-objection/SKILL.md')},
 'hello-world': {'description': 'A minim

## Routing — letting a model pick the skill

This is the part Claude Code does with a real LLM call: hand the model
every `(name, description)` pair plus the user's request, and let it
choose. Wired here through `brollm.BaseContract` to Bedrock's `converse`
API, model `google.gemma-3-4b-it` in `us-east-1`.

In [5]:
from bro_agent.llm import router, build_router_prompt, call_bedrock

chosen = router(user_request="find every place we log errors in dev.ipynb", registry=skill_registry)
print(chosen)
instructions = load_skill(chosen, skill_registry)
print(instructions)

grep-file
# Grep File

When invoked, you have access to two tools: `read_file(path)` and
`grep_file(pattern, path)`.

1. Identify the file path and the pattern from the request.
2. Call `grep_file(pattern, path)`.
3. Report each match as `line_number: line_text`. If there are no matches,
   say so plainly.

Do not read the whole file unless the user also asks for that.


In [5]:
r_prompt = build_router_prompt("find every place we log errors in dev.ipynb", skill_registry)
print(r_prompt)

Given this user request, pick the single best-matching skill by name.
Reply with only the skill name, nothing else.

Skills:
- grep-file: Search for a text pattern inside a specific file and return matching lines with their line numbers. Use this when the user names a file and a pattern/word/string to find within it.
- hello-world: A minimal example skill. Greets a person by name and explains what it just did. Use this to learn how skill discovery and loading works.
- word-count: Count words per line in a text file using the bundled count_words.py script. Use this when the user asks how many words are on each line, or wants a per-line word count, of a specific file.

User request: find every place we log errors in dev.ipynb


## Executing the chosen skill

Routing only produces a name. Executing means: ask the model, given the
skill's instructions, to decide which tool to call and with what
arguments; run that tool for real in the kernel; then feed the result
back so the model can write the final answer following the skill's
instructions. Three steps, so this is a `broflow` `Flow` — each step
decides its own next step, exactly like `PlanTask` here choosing `act`
when a tool is needed and `respond` when it isn't (e.g. `hello-world`
needs no tool at all).

In [6]:
from bro_agent.execution import run_script, execute_skill, skill_flow, PlanTask, ActTask, RespondTask

# The three-step plan/act/respond flow itself (PlanTask/ActTask/RespondTask,
# wired into a broflow.Flow) now lives in src/bro_agent/execution.py --
# import it instead of redefining it here. See that file's docstrings for
# how each step decides its own next step.

In [7]:
user_request = "grep for the word 'description' in bro_skills/grep-file/SKILL.md"

chosen = router(user_request=user_request, registry=skill_registry)
print("routed to:", chosen)

result = execute_skill(chosen, user_request, skill_registry)
print(skill_flow.trace)
print(result["answer"])

routed to: grep-file
[('plan', 'act'), ('act', 'respond'), ('respond', 'end')]
3: description: Search for a text pattern inside a specific file and return matching lines with their line numbers. Use this when the user names a file and a pattern/word/string to find within it.


In [11]:
print(load_skill(chosen, skill_registry))

# Grep File

When invoked, you have access to two tools: `read_file(path)` and
`grep_file(pattern, path)`.

1. Identify the file path and the pattern from the request.
2. Call `grep_file(pattern, path)`.
3. Report each match as `line_number: line_text`. If there are no matches,
   say so plainly.

Do not read the whole file unless the user also asks for that.


In [12]:
print(skill_dir(chosen, skill_registry))

D:\study-on-agent\bro_skills\grep-file


In [10]:
from bro_agent.execution import build_plan_prompt

_ = build_plan_prompt(load_skill(chosen, skill_registry), user_request, skill_dir(chosen, skill_registry))
print(_)

# Grep File

When invoked, you have access to two tools: `read_file(path)` and
`grep_file(pattern, path)`.

1. Identify the file path and the pattern from the request.
2. Call `grep_file(pattern, path)`.
3. Report each match as `line_number: line_text`. If there are no matches,
   say so plainly.

Do not read the whole file unless the user also asks for that.

Available tools:
- read_file
- grep_file

Decide which tool call satisfies the user's request, if any.
Reply with only a fenced json codeblock, either:
```json
{"tool": "grep_file", "args": {"pattern": "...", "path": "..."}}
```
or for a bundled script:
```json
{"tool": "run_script", "args": {"script": "count_words.py", "args": ["path/to/file"]}}
```
or, if no tool is needed:
```json
{"tool": null}
```

User request: grep for the word 'description' in bro_skills/grep-file/SKILL.md


In [13]:
TOOLS

{'read_file': <function bro_agent.execution.read_file(path: str) -> str>,
 'grep_file': <function bro_agent.execution.grep_file(pattern: str, path: str) -> list[tuple[int, str]]>}

In [8]:
result

{'instructions': '# Grep File\n\nWhen invoked, you have access to two tools: `read_file(path)` and\n`grep_file(pattern, path)`.\n\n1. Identify the file path and the pattern from the request.\n2. Call `grep_file(pattern, path)`.\n3. Report each match as `line_number: line_text`. If there are no matches,\n   say so plainly.\n\nDo not read the whole file unless the user also asks for that.',
 'user_request': "grep for the word 'description' in bro_skills/grep-file/SKILL.md",
 'skill_dir': WindowsPath('D:/study-on-agent/bro_skills/grep-file'),
 'tool': 'grep_file',
 'args': {'pattern': 'description', 'path': 'bro_skills/grep-file/SKILL.md'},
 'tool_result': [(3,
   'description: Search for a text pattern inside a specific file and return matching lines with their line numbers. Use this when the user names a file and a pattern/word/string to find within it.')],
 'answer': '3: description: Search for a text pattern inside a specific file and return matching lines with their line numbers. Use

## Bundled scripts — a skill can carry its own executable

`word-count/` bundles `count_words.py` next to its `SKILL.md`. `run_script`
runs it as a real subprocess (`sys.executable script.py arg1 arg2 ...`,
never `shell=True`) inside the skill's own folder, and only stdout/stderr
comes back — the script's source is never read into the model's context,
unlike `TOOLS` functions which run in-process. `build_plan_prompt` lists
whatever `*.py` files sit in a skill's folder so the model knows it can
call `run_script` at all.

In [14]:
# re-discover so the new word-count skill folder is picked up
skill_registry = discover_skills()

user_request = "count words per line in bro_skills/word-count/sample.txt"
chosen = router(user_request=user_request, registry=skill_registry)
print("routed to:", chosen)

result = execute_skill(chosen, user_request, skill_registry)
print(skill_flow.trace)
print(result["tool_result"])
print(result["answer"])

routed to: word-count
[('plan', 'act'), ('act', 'respond'), ('respond', 'end')]
1: 4
2: 2
3: 7

1: 4
2: 2
3: 7



In [17]:
from bro_agent.execution import build_plan_prompt

_ = build_plan_prompt(load_skill(chosen, skill_registry), user_request, skill_dir(chosen, skill_registry))
print(_)

# Word Count

This skill bundles a script instead of relying on in-process tools:
`count_words.py`, sitting next to this file.

When invoked:

1. Identify the file path from the request.
2. Run the tool call `{"tool": "run_script", "args": {"script": "count_words.py", "args": ["<path>"]}}`.
3. The script prints one `line_number: word_count` line per line of the
   file. Report those results back to the user.

Available tools:
- read_file
- grep_file
- run_script(script, args): runs one of this skill's bundled scripts (count_words.py); args is a list of command-line arguments

Decide which tool call satisfies the user's request, if any.
Reply with only a fenced json codeblock, either:
```json
{"tool": "grep_file", "args": {"pattern": "...", "path": "..."}}
```
or for a bundled script:
```json
{"tool": "run_script", "args": {"script": "count_words.py", "args": ["path/to/file"]}}
```
or, if no tool is needed:
```json
{"tool": null}
```

User request: count words per line in bro_skills/wor

In [18]:
print(load_skill(chosen, skill_registry))

# Word Count

This skill bundles a script instead of relying on in-process tools:
`count_words.py`, sitting next to this file.

When invoked:

1. Identify the file path from the request.
2. Run the tool call `{"tool": "run_script", "args": {"script": "count_words.py", "args": ["<path>"]}}`.
3. The script prints one `line_number: word_count` line per line of the
   file. Report those results back to the user.


In [19]:
print(skill_dir(chosen, skill_registry))

D:\study-on-agent\bro_skills\word-count


In [15]:
skill_registry

{'close-handoff': {'description': 'Terminal stage once a lead has agreed to move forward. Creates a handoff for a human rep -- never collects payment or signs anything itself. Use this while a lead is in the close_handoff stage of the funnel.',
  'stage': 'close_handoff',
  'path': WindowsPath('D:/study-on-agent/bro_skills/close-handoff/SKILL.md')},
 'grep-file': {'description': 'Search for a text pattern inside a specific file and return matching lines with their line numbers. Use this when the user names a file and a pattern/word/string to find within it.',
  'stage': None,
  'path': WindowsPath('D:/study-on-agent/bro_skills/grep-file/SKILL.md')},
 'handle-objection': {'description': 'Address a specific concern or pushback (price, fit, timing) a lead has raised. Use this while a lead is in the handle_objection stage of the funnel.',
  'stage': 'handle_objection',
  'path': WindowsPath('D:/study-on-agent/bro_skills/handle-objection/SKILL.md')},
 'hello-world': {'description': 'A minim